# Projekt 2 - Low light image enhancement: baseline, trening, ewaluacja

Ten notebook działa **w Colabie i lokalnie** (na komputerze z GPU). Przygotowanie danych jest w `00_data_preparation.ipynb`.

Plik zawiera części takie jak:
1. **baseline:** wyrównanie histogramu na zbiorze testowym (20 ostatnich obrazów),
2. **trening** sieci U-Net (jeśli model już istnieje w folderze `runs`, trening jest pomijany),
3. **ewaluacja** na zbiorze testowym: tabela `baseline` / `network`, wizualizacje, analiza przepaleń.

**Uruchomienie Lokalne:** umieść ten notebook w folderze z plikami `.py` i `RAISE_258.csv`, a obrazy (foldery `gt` i `dark`) w folderze `lowlight_data`.
Potrzebny jest PyTorch z obsługą karty (instalacja ze strony pytorch.org); resztę bibliotek notebook doinstaluje sam.

## 1. Ustawienia środowiska wykonawczego

In [ ]:
import os, sys

IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    CODE_DIR = '/content/drive/MyDrive/sigk_lowlight'     # .py files, the csv, models and results (on Drive)
    DATA_DIR = '/content/lowlight_data'                    # local copy of the images (faster than Drive)
else:
    CODE_DIR = '.'                                         # folder with the .py files and the csv
    DATA_DIR = 'lowlight_data'                             # folder with the subfolders gt/ and dark/

CSV_PATH = os.path.join(CODE_DIR, 'RAISE_258.csv')
RUNS_DIR = os.path.join(CODE_DIR, 'runs')                  # trained models and training history
RESULTS_DIR = os.path.join(CODE_DIR, 'results')            # tables and figures
os.makedirs(RESULTS_DIR, exist_ok=True)
sys.path.append(CODE_DIR)

print('Colab:', IN_COLAB, '| csv exists:', os.path.exists(CSV_PATH))

In [ ]:
# install missing libraries (PyTorch itself has to be installed by hand locally)
import subprocess

def ensure(package, module):
    try:
        __import__(module)
    except ImportError:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', package])

for package, module in [('lpips', 'lpips'), ('torchmetrics', 'torchmetrics'), ('pandas', 'pandas'),
                        ('opencv-python', 'cv2'), ('matplotlib', 'matplotlib')]:
    ensure(package, module)

In [ ]:
import json
import shutil

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
import lpips
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader

from dataset import LowLightDataset, read_names, split_names
from baselines import histogram_equalization
from metrics import all_metrics
from models import UNet
from train import train

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', device, '|', torch.cuda.get_device_name(0) if device == 'cuda' else 'NO GPU - training will be very slow')

### Dane
W Colabie kopiujemy obrazy z Dysku na lokalny dysk (czytanie z Dysku w każdej epoce jest bardzo wolne). Lokalnie obrazy muszą być w `DATA_DIR`.

In [ ]:
if IN_COLAB and not os.path.exists(DATA_DIR):
    for name in ('lowlight_data', 'data'):                 # the two names of the backup folder we used
        source = os.path.join(CODE_DIR, name)
        if os.path.exists(os.path.join(source, 'gt')):
            shutil.copytree(source, DATA_DIR)
            break

assert os.path.exists(os.path.join(DATA_DIR, 'gt')), f'No images in {DATA_DIR} (folders gt/ and dark/ are needed)'
print('gt:', len(os.listdir(os.path.join(DATA_DIR, 'gt'))), '| dark:', len(os.listdir(os.path.join(DATA_DIR, 'dark'))))

train_names, val_names, test_names = split_names(read_names(CSV_PATH))
print('train:', len(train_names), '| val:', len(val_names), '| test:', len(test_names))

## 2. Funkcje do ewaluacji
* `predict_padded`: U-Net trzy razy zmniejsza obraz o połowę, więc wysokość i szerokość muszą dzielić się przez 8. Zdjęcia testowe (np. 1640 x 2474) tego nie spełniają, więc dodajemy brzeg (odbicie lustrzane), uruchamiamy sieć i przycinamy wynik.
* `run_test`: dla każdego obrazu testowego liczy PSNR, SSIM i LPIPS dla: ciemnego wejścia, baseline'u i sieci (całe obrazy, nie wycinki).

In [ ]:
lpips_model = lpips.LPIPS(net='alex').to(device)
test_ds = LowLightDataset(DATA_DIR, test_names, mode='test')
test_dl = DataLoader(test_ds, batch_size=1, shuffle=False)


def predict_padded(model, x, multiple=8):
    _, _, h, w = x.shape
    pad_h = (-h) % multiple
    pad_w = (-w) % multiple
    x_pad = F.pad(x, (0, pad_w, 0, pad_h), mode='reflect')      # (left, right, top, bottom)
    return model(x_pad)[:, :, :h, :w]


def load_unet(name):
    model = UNet().to(device)
    model.load_state_dict(torch.load(os.path.join(RUNS_DIR, name, 'best.pt'), map_location=device, weights_only=True))
    model.eval()
    return model


def run_test(models):
    # models: dictionary {name: trained model}; returns metrics of every test image and every method
    rows = []
    with torch.no_grad():
        for image_name, (dark, gt) in zip(test_names, test_dl):
            outputs = {'dark input': dark, 'baseline': histogram_equalization(dark)}
            for model_name, model in models.items():
                outputs[f'network ({model_name})'] = predict_padded(model, dark.to(device)).cpu()
            for method, pred in outputs.items():
                m = all_metrics(pred.clamp(0, 1).to(device), gt.to(device), lpips_model)
                rows.append({'image': image_name, 'method': method, 'PSNR': m['PSNR'].item(),
                             'SSIM': m['SSIM'].item(), 'LPIPS': m['LPIPS'].item()})
    return pd.DataFrame(rows)

## 3. Baseline (przed treningiem)
Wyrównanie histogramu na kanale jasności, liczone na 20 obrazach testowych.

In [ ]:
per_image = run_test({})
table = per_image.groupby('method', sort=False)[['PSNR', 'SSIM', 'LPIPS']].mean()
table.round(4)

## 4. Trening
Osobny model dla każdej wartości wagi LPIPS w funkcji straty (`0.0` = tylko L1). Wynik każdego treningu trafia do `runs/<nazwa>/` (`best.pt`, `history.json`).
**Jeśli model o danej nazwie już istnieje, trening jest pomijany**.
Aby wytrenować od nowa, usuń folder modelu albo ustaw `RETRAIN = True`.

In [ ]:
EPOCHS = 75
RETRAIN = False
experiments = {'unet_l1': 0.0, 'unet_lpips': 0.05}      # name -> weight of the LPIPS part of the loss

for name, lpips_weight in experiments.items():
    out_dir = os.path.join(RUNS_DIR, name)
    if os.path.exists(os.path.join(out_dir, 'best.pt')) and not RETRAIN:
        print(f'{name}: model already exists, training skipped')
        continue
    print(f'=== training {name} ===')
    train(DATA_DIR, CSV_PATH, out_dir, epochs=EPOCHS, lpips_weight=lpips_weight)

### Krzywe uczenia

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
model_names = ['unet_l1', 'unet_lpips']

for ax, name in zip(axes, model_names):
    with open(os.path.join(RUNS_DIR, name, 'history.json'), 'r', encoding='utf-8') as f:
        hist = json.load(f)

    epochs = [h['epoch'] for h in hist]
    train_loss = [h['loss'] for h in hist]
    val_loss = [h['val_loss'] for h in hist]
    best_ep = min(hist, key=lambda h: h['val_loss'])

    ax.plot(epochs, train_loss, label='train loss', linewidth=2)
    ax.plot(epochs, val_loss, label='val loss', linewidth=2)
    ax.axvline(best_ep['epoch'], color='gray', linestyle='--', alpha=0.6,
               label=f"min val loss (ep. {best_ep['epoch']})")
    ax.set_title(f'{name} — train vs validation loss')
    ax.set_xlabel('epoch')
    ax.set_ylabel('loss')
    ax.grid(True, alpha=0.3)
    ax.legend()

plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, 'training_curves.png'), dpi=120)
plt.show()


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for name in experiments:
    path = os.path.join(RUNS_DIR, name, 'history.json')
    if not os.path.exists(path):
        continue
    h = json.load(open(path, encoding='utf-8'))
    val = [e['val_psnr'] for e in h]
    best = int(np.argmax(val))
    axes[0].plot([e['loss'] for e in h], label=name)
    axes[1].plot(val, label=f'{name} (best: epoch {best + 1}, {val[best]:.2f} dB)')
axes[0].set_title('train loss')
axes[1].set_title('validation PSNR [dB]')
for ax in axes:
    ax.set_xlabel('epoch')
    ax.legend()
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, 'training_curves.png'), dpi=150)
plt.show()

## 5. Wyniki na zbiorze testowym
Tabela dla 20 ostatnich obrazów (całe obrazy). PSNR i SSIM: wyżej = lepiej, LPIPS: niżej = lepiej.

In [ ]:
models = {name: load_unet(name) for name in experiments if os.path.exists(os.path.join(RUNS_DIR, name, 'best.pt'))}
per_image = run_test(models)
per_image.to_csv(os.path.join(RESULTS_DIR, 'per_image.csv'), index=False)

table = per_image.groupby('method', sort=False)[['PSNR', 'SSIM', 'LPIPS']].mean()
table.to_csv(os.path.join(RESULTS_DIR, 'results.csv'))
table.round(4)

## 6. Wizualizacje
Ciemny obraz na wejściu, baseline, sieć i obraz wzorcowy dla najlepszego i najgorszego obrazu testowego oraz dla pierwszego obrazu.

In [ ]:
main_name = list(models)[0]                                     # which network to show (change if needed)
model = models[main_name]
net = per_image[per_image.method == f'network ({main_name})'].reset_index(drop=True)
picked = [int(net['PSNR'].idxmax()), int(net['PSNR'].idxmin()), 0]
titles = ['best', 'worst', 'first']


def small(t):
    return t[0].clamp(0, 1).permute(1, 2, 0).cpu().numpy()[::3, ::3]      # every 3rd pixel, the figure stays small


fig, axes = plt.subplots(len(picked), 4, figsize=(20, 4 * len(picked)))
for r, i in enumerate(picked):
    dark, gt = test_ds[i]
    dark, gt = dark.unsqueeze(0), gt.unsqueeze(0)
    with torch.no_grad():
        out = predict_padded(model, dark.to(device)).cpu()
    panels = [('dark input', dark), ('baseline', histogram_equalization(dark)),
              (f'network ({main_name})', out), ('ground truth', gt)]
    for col, (title, img) in enumerate(panels):
        axes[r, col].imshow(small(img))
        axes[r, col].set_title(f'{titles[r]}: {title}' if col == 0 else title)
        axes[r, col].axis('off')
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, 'examples.png'), dpi=100)
plt.show()

In [ ]:
net = per_image[per_image.method == 'network (unet_lpips)'].reset_index(drop=True)
median_idx = int((net['PSNR'] - net['PSNR'].median()).abs().idxmin())

picked = [int(net['PSNR'].idxmax()), int(net['PSNR'].idxmin()), median_idx]
titles = ['best', 'worst', 'median']

def small(t):
    return t[0].clamp(0, 1).permute(1, 2, 0).cpu().numpy()[::3, ::3]

fig, axes = plt.subplots(len(picked), 5, figsize=(25, 4 * len(picked)))
for r, i in enumerate(picked):
    dark, gt = test_ds[i]
    dark, gt = dark.unsqueeze(0), gt.unsqueeze(0)
    with torch.no_grad():
        out_l1 = predict_padded(models['unet_l1'], dark.to(device)).cpu()
        out_lpips = predict_padded(models['unet_lpips'], dark.to(device)).cpu()
        
    panels = [
        ('dark input', dark),
        ('baseline', histogram_equalization(dark)),
        ('unet_l1', out_l1),
        ('unet_lpips', out_lpips),
        ('ground truth', gt)
    ]
    for col, (title, img) in enumerate(panels):
        axes[r, col].imshow(small(img))
        axes[r, col].set_title(f'{titles[r]}: {title}' if col == 0 else title)
        axes[r, col].axis('off')

plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, 'examples_comparison.png'), dpi=300)
plt.show()

In [ ]:
#wybór: średni PSNR obu modeli na każdym ze zdjęć testowych
m_l1 = per_image[per_image.method == 'network (unet_l1)'].reset_index(drop=True)
m_lpips = per_image[per_image.method == 'network (unet_lpips)'].reset_index(drop=True)

mean_psnr = (m_l1['PSNR'] + m_lpips['PSNR']) / 2.0
median_idx = int((mean_psnr - mean_psnr.median()).abs().idxmin())

picked = [int(mean_psnr.idxmax()), int(mean_psnr.idxmin()), median_idx]
titles = ['best (avg)', 'worst (avg)', 'median (avg)']

def small(t):
    return t[0].clamp(0, 1).permute(1, 2, 0).cpu().numpy()[::3, ::3]

fig, axes = plt.subplots(len(picked), 5, figsize=(25, 4.2 * len(picked)))
for r, i in enumerate(picked):
    dark, gt = test_ds[i]
    dark, gt = dark.unsqueeze(0), gt.unsqueeze(0)
    with torch.no_grad():
        out_l1 = predict_padded(models['unet_l1'], dark.to(device)).cpu()
        out_lpips = predict_padded(models['unet_lpips'], dark.to(device)).cpu()

    panels = [
        (f"{titles[r]} [{test_names[i]}]\ndark input", dark),
        ("baseline", histogram_equalization(dark)),
        (f"unet_l1\n(PSNR: {m_l1.loc[i, 'PSNR']:.2f} dB | LPIPS: {m_l1.loc[i, 'LPIPS']:.3f})", out_l1),
        (f"unet_lpips\n(PSNR: {m_lpips.loc[i, 'PSNR']:.2f} dB | LPIPS: {m_lpips.loc[i, 'LPIPS']:.3f})", out_lpips),
        ("ground truth", gt)
    ]
    for col, (title, img) in enumerate(panels):
        axes[r, col].imshow(small(img))
        axes[r, col].set_title(title, fontsize=10)
        axes[r, col].axis('off')

plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, 'examples_comparison.png'), dpi=300)
plt.show()

In [ ]:
import matplotlib.patches as patches

img_idx = 0       # indeks zdjęcia ze zbioru testowego (np. 0 to bilet rc589a8d6t)
y, x = 330, 370   # współrzędne lewego górnego rogu wycinka
crop_size = 256   # rozmiar powiększonego fragmentu w pikselach

dark, gt = test_ds[img_idx]
dark_b, gt_b = dark.unsqueeze(0), gt.unsqueeze(0)

with torch.no_grad():
    out_l1 = predict_padded(models['unet_l1'], dark_b.to(device)).cpu()
    out_lpips = predict_padded(models['unet_lpips'], dark_b.to(device)).cpu()

def get_full(t):
    return t[0].clamp(0, 1).permute(1, 2, 0).numpy()

def get_crop(t):
    return t[0, :, y:y+crop_size, x:x+crop_size].clamp(0, 1).permute(1, 2, 0).numpy()

panels = [
    ('dark input', dark_b),
    ('unet_l1', out_l1),
    ('unet_lpips', out_lpips),
    ('ground truth', gt_b)
]

fig, axes = plt.subplots(2, 4, figsize=(18, 9))

for col, (title, tensor_img) in enumerate(panels):
    # Wiersz 0: pełne zdjęcie + czerwona ramka wskazująca wycinek
    axes[0, col].imshow(get_full(tensor_img))
    rect = patches.Rectangle(
        (x, y), crop_size, crop_size,
        linewidth=2, edgecolor='red', facecolor='none'
    )
    axes[0, col].add_patch(rect)
    axes[0, col].set_title(f"{title} (całość)")
    axes[0, col].axis('off')

    # Wiersz 1: przybliżenie 1:1 na wybrany wycinek
    axes[1, col].imshow(get_crop(tensor_img))
    axes[1, col].set_title(f"{title} (wycinek {crop_size}x{crop_size})")
    axes[1, col].axis('off')

plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, f'full_and_crop_img{img_idx}.png'), dpi=150)
plt.show()

In [ ]:
indices = [5, 4]
row_labels = ['Bardzo korzystny (niższa ekspozycja GT)', 'Mało korzystny (wysoka ekspozycja GT)']

net_l1 = per_image[per_image.method == 'network (unet_l1)'].reset_index(drop=True)
net_lpips = per_image[per_image.method == 'network (unet_lpips)'].reset_index(drop=True)

def small(t):
    return t[0].clamp(0, 1).permute(1, 2, 0).cpu().numpy()[::3, ::3]

fig, axes = plt.subplots(2, 4, figsize=(20, 8))

for r, idx in enumerate(indices):
    dark, gt = test_ds[idx]
    dark_b, gt_b = dark.unsqueeze(0), gt.unsqueeze(0)

    with torch.no_grad():
        out_l1 = predict_padded(models['unet_l1'], dark_b.to(device)).cpu()
        out_lpips = predict_padded(models['unet_lpips'], dark_b.to(device)).cpu()

    psnr_l1 = net_l1.loc[idx, 'PSNR']
    psnr_lpips = net_lpips.loc[idx, 'PSNR']

    panels = [
        (f"{row_labels[r]}\n[{idx}: {test_names[idx]}] — dark input", dark_b),
        (f"unet_l1 (PSNR: {psnr_l1:.2f} dB)", out_l1),
        (f"unet_lpips (PSNR: {psnr_lpips:.2f} dB)", out_lpips),
        ("ground truth", gt_b)
    ]

    for col, (title, img) in enumerate(panels):
        axes[r, col].imshow(small(img))
        axes[r, col].set_title(title, fontsize=11)
        axes[r, col].axis('off')

plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, 'coins_exposure_comparison.png'), dpi=300)
plt.show()

## 7. Przypadki specjalne - przepalenia
Dla każdego obrazu testowego liczymy, jaki odsetek pikseli obrazu wzorcowego jest przepalony (którykolwiek kanał >= 250), i zestawiamy to z PSNR.

In [ ]:
saturated = [float((test_ds.gt[i].max(axis=2) >= 250).mean()) * 100 for i in range(len(test_ds))]

summary = pd.DataFrame({'image': test_names, 'saturated [%]': saturated})
for method in per_image.method.unique():
    values = per_image[per_image.method == method].reset_index(drop=True)['PSNR']
    summary[f'PSNR {method}'] = values
summary.sort_values('saturated [%]', ascending=False).round(2)

In [ ]:
summary = pd.DataFrame({'image': test_names})
for method in per_image.method.unique():
    values = per_image[per_image.method == method].reset_index(drop=True)['PSNR']
    summary[f'PSNR {method}'] = values

sort_col = 'PSNR network (unet_lpips)'  # możesz zmienić na 'PSNR network (unet_l1)'
ranked = summary.sort_values(sort_col, ascending=False).round(2)

print("=== 10 NAJLEPSZYCH ZDJĘĆ ===")
display(ranked.head(10))

print("\n=== 10 NAJGORSZYCH ZDJĘĆ ===")
display(ranked.tail(10))